## MUST HAVE 텐초의 파이토치

In [ ]:
import torch

a = torch.tensor([1,2,3])
b = torch.tensor([4,5,6])
c = a + b
print(c)

tensor([5, 7, 9])


In [ ]:
# class magic methods

class Magic:
    def __init__(self):
        print('__init__ called')
        self.numbers = [n for n in range(10)]

    def __len__(self):
        print('__len__ called')
        return len(self.numbers)

    def __getitem__(self, index):
        print('__getitem__ called')
        return self.numbers[index]

    def __setitem__(self, index, value):
        print('__setitem__ called')
        self.numbers[index] = value

m = Magic()
print(len(m))
print(m[1])
m[1] = 100
print(m[1] == m.numbers[1])     # 이건 뭐야?

__init__ called
__len__ called
10
__getitem__ called
1
__setitem__ called
__getitem__ called
True


위의 예에서는 배열이 아닌 타입의 m이라는 인스턴스를 마치 배열처럼 사용하고 있는데, 이런 방식은 파이토치에서 Dataset과 같이 온전히 데이터만 담고 있는 클래스에서는 직관적인 표현이라서 이런 방식의 사용이 권장된다고 한다.

### PyTorch 커스텀 데이터셋 예시
파이토치에서는 아래와 같이 `Dataset`을 상속받아 인덱스로 데이터에 접근할 수 있게 만듭니다. 이렇게 하면 `DataLoader`라는 도구가 이 데이터를 자동으로 섞고 나눠서 모델에 넣어줄 수 있습니다.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 데이터를 가져올 때 전처리를 하거나 변형을 줄 수도 있습니다.
        return self.data[idx]

# 데이터셋 생성
raw_data = torch.randn(10, 3) # 10개의 샘플(각 샘플이 3개의 특징을 가지고 있음)
dataset = MyDataset(raw_data)

# 이제 dataset[0] 처럼 인덱스로 바로 접근 가능합니다.
print(f"첫 번째 데이터: {dataset[0]}")

# DataLoader가 이 매직 메서드들을 활용해 데이터를 관리합니다.
loader = DataLoader(dataset, batch_size=2, shuffle=True)
for batch in loader:
    print("배치 데이터:", batch)
    break

첫 번째 데이터: tensor([ 1.2155,  1.7472, -0.8962])
배치 데이터: tensor([[ 0.7083, -0.6323, -0.2288],
        [-0.5418,  0.5623, -1.2990]])


In [ ]:
def explain_vargs(*args):
    print(f"전달된 인자들: {args}")
    print(f"인자의 타입: {type(args)}")
    print(f"인자의 개수: {len(args)}")
    print("-" * 20)

# 1개, 2개, 3개의 인자를 각각 넣어봅시다
explain_vargs(10)
explain_vargs(10, 3)
explain_vargs(2, 3, 4, 5)

전달된 인자들: (10,)
인자의 타입: <class 'tuple'>
인자의 개수: 1
--------------------
전달된 인자들: (10, 3)
인자의 타입: <class 'tuple'>
인자의 개수: 2
--------------------
전달된 인자들: (2, 3, 4, 5)
인자의 타입: <class 'tuple'>
인자의 개수: 4
--------------------


In [ ]:
my_shape = [10, 3, 5]

print("1. 그냥 넣었을 때:")
explain_vargs(my_shape)

print("\n2. 별(*)을 붙여서 넣었을 때 (Unpacking):")
explain_vargs(*my_shape)

차이가 보이시나요?
1. 그냥 넣으면 리스트 전체가 하나의 인자로 취급되어 `([10, 3, 5],)` 형태의 튜플이 됩니다.
2. 하지만 `*my_shape`라고 쓰면 리스트의 알맹이들이 각각 10, 3, 5로 풀려서 전달됩니다.

`torch.randn(*my_shape)`라고 쓰면 리스트에 담긴 숫자로 텐서 모양을 만들 수 있게 되는 거죠!

위의 예시를 실행해 보면, 우리가 인자를 쉼표로 나열만 해도 함수 안에서는 `args`라는 하나의 튜플로 묶이는 걸 볼 수 있습니다.

`torch.randn(10, 3)`도 내부적으로는 이 `*size`를 통해 `(10, 3)`이라는 튜플을 전달받아 텐서의 모양을 결정하는 것이죠. 만약 이미 튜플이나 리스트로 가지고 있는 값을 가변 인자 함수에 넣고 싶을 때는 어떻게 해야 할까요? (힌트: 다시 별을 사용합니다!)

In [ ]:
raw_data

tensor([[ 1.2155,  1.7472, -0.8962],
        [-0.7312,  0.2140,  0.9834],
        [-0.5418,  0.5623, -1.2990],
        [-0.9285,  1.1368,  0.9650],
        [ 0.8780,  1.7235,  1.5230],
        [-0.4553, -1.5383, -0.7456],
        [-0.4274,  0.2439, -2.7288],
        [-0.0655,  0.9389,  0.3869],
        [ 0.7083, -0.6323, -0.2288],
        [ 0.6939,  1.5322,  0.6785]])

In [ ]:
# raw_data의 통계값 확인
print(f"최소값: {raw_data.min()}")
print(f"최대값: {raw_data.max()}")
print(f"평균: {raw_data.mean()}")
print(f"표준편차: {raw_data.std()}")

최소값: -2.7288331985473633
최대값: 1.7472375631332397
평균: 0.16376608610153198
표준편차: 1.0766716003417969


### 통계적 원리: 정규 분포의 선형 변환
`torch.randn`은 평균이 0이고 표준편차가 1인 표준 정규 분포(Standard Normal Distribution)를 따르는 난수값을 생성합니다.

즉, 평균은 0이고, 표준편차는 1인 분포입니다. 수학적으로 값의 범위에 제한은 없지만, 약 68%의 데이터는 -1에서 1 사이에, 95%는 -2에서 2 사이에, 99.7%는 -3에서 3 사이에 존재하게 됩니다.

이를 수학적으로 표현하면 $Z \sim N(0, 1)$ 입니다. 여기에 임의의 상수 $a$와 $b$를 이용하여 $X = aZ + b$라는 새로운 변수를 만들면, $X$의 통계적 특성은 다음과 같습니다.

*   **평균:** $E(aZ + b) = aE(Z) + b = a(0) + b = b$
*   **표준편차:** $SD(aZ + b) = \sqrt{Var(aZ + b)} = \sqrt{a^2 Var(Z)} = |a| \cdot 1 = |a|$

따라서 `samples = torch.randn(size) * std + mean` 식은:
1. `std`를 곱하여 분포의 퍼짐 정도(표준편차)를 조절하고,
2. `mean`을 더하여 분포의 중심 위치(평균)를 이동시키는 원리를 이용한 것입니다.

In [ ]:
import torch

# 설정값
target_mean = 2.0
target_std = 1.5

# 10,000개의 샘플을 생성하여 통계적으로 확인
large_samples = torch.randn(10000) * target_std + target_mean

print(f"생성된 샘플의 평균: {large_samples.mean():.4f} (목표: {target_mean})")
print(f"생성된 샘플의 표준편차: {large_samples.std():.4f} (목표: {target_std})")

생성된 샘플의 평균: 2.0188 (목표: 2.0)
생성된 샘플의 표준편차: 1.5039 (목표: 1.5)
